# MLB Edge Finder — Exploration

Interactive walkthrough of the pipeline stages.

In [ ]:
import logging
from datetime import date

from mlb_edge_finder import config

from pybaseball import cache
cache.enable()

config.setup_logging(level=logging.INFO)

## 1. Fetch Odds

In [ ]:
from mlb_edge_finder import odds_ingestion

game_date = date.today()
odds_df = odds_ingestion.fetch_odds(game_date, force=True, debug=True)
odds_df.head()

## 2. Fetch Stats

In [ ]:
from mlb_edge_finder import stats_ingestion

stats_df = stats_ingestion.fetch_stats(game_date)
stats_df.head()

## 3. Build Features

In [ ]:
from mlb_edge_finder import features
from mlb_edge_finder.pitcher_ingestion import fetch_pitcher_stats

# Fetch pitcher stats for today's season (cached after first run)
# Required before build_features() — fetches all pitcher season stats
fetch_pitcher_stats(game_date)

features_df = features.build_features(game_date)
features_df.head()
features_df[features_df.columns[:10]].head()


## 4a. Historical Ingestion

Fetch completed regular season results for each training season via `statsapi`.

In [ ]:
from mlb_edge_finder import historical_ingestion

# Fetch (or load cached) results for a single season
hist_2024 = historical_ingestion.fetch_historical(2024)
print(f"{len(hist_2024)} games")
hist_2024.head()

In [ ]:
# Concatenate all training seasons (2023, 2024, 2025)
all_hist = historical_ingestion.fetch_all_historical()
print(f"{len(all_hist)} total games")
all_hist.groupby(all_hist['game_date'].str[:4])['home_win'].agg(['count', 'mean'])

## 4b. Training Data

Join end-of-season team stats + rolling window stats (last 15 games: runs scored, 
runs allowed, win %, run diff) to each game row to produce the model training set.

Run with `force=True` to rebuild the cache with rolling columns included.

In [ ]:
from mlb_edge_finder import training_data

seasons = [2019, 2021, 2022, 2023, 2024, 2025]
# force=True rebuilds cache to include all 6 seasons (2019, 2021-2025; 2020 skipped — 60-game anomaly)
training_df = training_data.build_training_set(seasons, force=True)
print(f"{len(training_df)} rows, {len(training_df.columns)} columns")
training_df.head()


In [ ]:
# Class balance and missing-value check
# Note: rolling stat columns will show NaN for the first game of each season
# per team (no prior games to average) — this is expected; XGBoost handles NaN natively.
print("home_win distribution:")
print(training_df['home_win'].value_counts())
print()
nulls = training_df.isnull().sum()
print("Null counts:", nulls[nulls > 0].to_dict() or "none")

## 4c. Model Training

Train an XGBoost classifier on the training set, evaluate it, and persist the model and metrics.

In [ ]:
from datetime import date
from mlb_edge_finder import model

clf, X_test, y_test = model.train(training_df)
print(f"Test set size: {len(X_test)} games")
print(f"Features used: {list(X_test.columns)}")

In [ ]:
baseline_clf, _, _ = model.train_baseline(training_df)
print("Baseline (logistic regression) trained.")

In [ ]:
import pandas as pd

xgb_metrics = model.evaluate(clf, X_test, y_test)
lr_metrics = model.evaluate(baseline_clf, X_test, y_test)

comparison = pd.DataFrame(
    {"XGBoost": xgb_metrics, "LogisticRegression": lr_metrics},
    index=xgb_metrics.keys(),
)
print(comparison.to_string())

model.save_model(clf, xgb_metrics, date.today())

In [ ]:
loaded_clf = model.load_model(date.today())
print("Model reloaded successfully.")
print(f"Sample predictions: {loaded_clf.predict(X_test[:3])}")

## 5. Find Edges

Run inference on today's features and flag games where the model finds positive expected value.

In [ ]:
from mlb_edge_finder import edge_finder

edges = edge_finder.find_edges(features_df, clf, game_date)
print(f"{len(edges)} edge(s) found for {game_date}")
edges

### 5b. Full Pipeline (end-to-end)

`pipeline.run()` drives all five stages — odds fetch, stats fetch, feature build, model load, edge find — in one call.

In [ ]:
from mlb_edge_finder import pipeline

# Runs end-to-end for today: fetch odds, stats, build features, load model, find edges
pipeline_edges = pipeline.run(game_date)
pipeline_edges

## 7. Starting Pitcher Features

Fetch season-to-date individual pitcher stats via the MLB Stats API, and look up today's probable starters.

In [ ]:
from mlb_edge_finder.pitcher_ingestion import (
    fetch_pitcher_stats,
    load_cached_pitcher_stats,
    fetch_probable_starters,
)
from datetime import date

# Fetch season pitcher stats (cached after first run)
# Columns: pitcher_id, pitcher_name, era, whip, k_per_9, bb_per_9, ip, fip_computed
snapshot_date = date(2025, 9, 28)
pitcher_df = fetch_pitcher_stats(snapshot_date)
print(f"Fetched {len(pitcher_df)} pitchers")
pitcher_df.head(10)


In [ ]:
# Fetch today's probable starters (live call, not cached — starters can change day-of)
today = date.today()
starters_df = fetch_probable_starters(today)
print(f"Probable starters for {today}: {len(starters_df)} games")
starters_df


In [ ]:
# Verify pitcher sp columns appear in training set
sp_cols = [c for c in training_df.columns if c.startswith('home_sp_') or c.startswith('away_sp_')]
print(f"Pitcher feature columns: {sp_cols}")
training_df[['home_starter_name', 'away_starter_name'] + sp_cols[:4]].head()
